# Tugas Mandiri Pertemuan 9: Katalog Menu Kopi Nusantara di MongoDB

**Nama:** Hilmi Mufid | **NPM:** 2505060046 | **Program Studi:** S1 Teknologi Informasi

**Skenario:** Platform Kopi Nusantara mulai menyimpan katalog menu di MongoDB karena tiap kategori menu memiliki atribut yang berbeda. Notebook ini membangun, menelusuri, memperbarui, dan menganalisis katalog tersebut dengan PyMongo dan PySpark.

Pastikan MongoDB aktif sebelum menjalankan notebook ini (`sudo systemctl status mongod`).

## A. Membangun Katalog (Create)

Database `kopi_nusantara`, collection `menu`, berisi 14 dokumen: 6 menu Kopi (field `tingkat_kepahitan`), 5 menu Makanan (field `pedas_level`), dan 3 menu Non-Kopi (field `tingkat_manis`). Beberapa menu Kopi juga memiliki `rekomendasi_barista`, sehingga bentuk dokumen sengaja tidak seragam.

In [1]:
from pymongo import MongoClient

# Koneksi ke MongoDB lokal, lalu membuat database dan collection baru
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Membersihkan data lama agar notebook aman dijalankan ulang (Run All)
koleksi_menu.delete_many({})

# 14 dokumen menu, field-nya berbeda sesuai kategori:
# Kopi memiliki tingkat_kepahitan, Makanan memiliki pedas_level, Non-Kopi memiliki tingkat_manis
koleksi_menu.insert_many([
    # Kategori Kopi
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 50, "rekomendasi_barista": True},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 60},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 70, "rekomendasi_barista": True},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 100, "rekomendasi_barista": True},
    {"nama_menu": "Latte", "harga": 27000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 80},
    {"nama_menu": "Kopi Tubruk", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 40},
    # Kategori Makanan
    {"nama_menu": "Roti Bakar", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 45},
    {"nama_menu": "Kentang Goreng", "harga": 20000, "kategori": "Makanan", "pedas_level": 0, "stok": 60},
    {"nama_menu": "Mie Goreng Pedas", "harga": 28000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Nasi Ayam Sambal Matah", "harga": 32000, "kategori": "Makanan", "pedas_level": 2, "stok": 35},
    {"nama_menu": "Pisang Goreng Keju", "harga": 17000, "kategori": "Makanan", "pedas_level": 0, "stok": 50},
    # Kategori Non-Kopi
    {"nama_menu": "Es Teh Lemon", "harga": 12000, "kategori": "Non-Kopi", "tingkat_manis": 3, "stok": 120},
    {"nama_menu": "Matcha Latte", "harga": 28000, "kategori": "Non-Kopi", "tingkat_manis": 4, "stok": 55},
    {"nama_menu": "Cokelat Panas", "harga": 24000, "kategori": "Non-Kopi", "tingkat_manis": 5, "stok": 65},
])

print("Koneksi berhasil. Daftar database:", client.list_database_names())
print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Koneksi berhasil. Daftar database: ['admin', 'config', 'kopi_nusantara', 'local', 'praktikum_bigdata', 'praktikum_bigdata0046']
Jumlah dokumen di collection 'menu': 14


## B. Kueri Bertingkat (Read)

Tiga kueri: (1) menu Kopi dengan kepahitan di atas 3, (2) menu dengan harga antara 15000 dan 30000, (3) menu yang tidak memiliki field `pedas_level`.

In [2]:
# Kueri 1: menu Kopi dengan tingkat_kepahitan di atas 3
print("1. Menu Kopi dengan tingkat_kepahitan di atas 3:")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  * {m['nama_menu']} (kepahitan {m['tingkat_kepahitan']}, Rp{m['harga']:,})")

print()

# Kueri 2: menu dengan harga di antara 15000 dan 30000 (batas atas dan bawah ikut dihitung)
print("2. Menu dengan harga antara 15000 dan 30000:")
for m in koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}}):
    print(f"  * {m['nama_menu']} ({m['kategori']}, Rp{m['harga']:,})")

print()

# Kueri 3: menu yang sama sekali tidak memiliki field pedas_level
print("3. Menu yang tidak memiliki field pedas_level:")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  * {m['nama_menu']} ({m['kategori']})")

1. Menu Kopi dengan tingkat_kepahitan di atas 3:
  * Espresso (kepahitan 5, Rp18,000)
  * Americano (kepahitan 4, Rp20,000)
  * Kopi Tubruk (kepahitan 5, Rp15,000)

2. Menu dengan harga antara 15000 dan 30000:
  * Espresso (Kopi, Rp18,000)
  * Americano (Kopi, Rp20,000)
  * Cappuccino (Kopi, Rp25,000)
  * Kopi Susu Gula Aren (Kopi, Rp22,000)
  * Latte (Kopi, Rp27,000)
  * Kopi Tubruk (Kopi, Rp15,000)
  * Roti Bakar (Makanan, Rp15,000)
  * Kentang Goreng (Makanan, Rp20,000)
  * Mie Goreng Pedas (Makanan, Rp28,000)
  * Pisang Goreng Keju (Makanan, Rp17,000)
  * Matcha Latte (Non-Kopi, Rp28,000)
  * Cokelat Panas (Non-Kopi, Rp24,000)

3. Menu yang tidak memiliki field pedas_level:
  * Espresso (Kopi)
  * Americano (Kopi)
  * Cappuccino (Kopi)
  * Kopi Susu Gula Aren (Kopi)
  * Latte (Kopi)
  * Kopi Tubruk (Kopi)
  * Es Teh Lemon (Non-Kopi)
  * Matcha Latte (Non-Kopi)
  * Cokelat Panas (Non-Kopi)


## C. Pembaruan Data (Update)

Field `status` ditambahkan ke seluruh dokumen dengan `update_many()`, lalu dua menu diubah menjadi `Habis` dengan `update_one()`.

In [3]:
# update_many(): menambahkan field status = "Tersedia" ke SELURUH dokumen
hasil = koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
print("Dokumen yang diberi status 'Tersedia':", hasil.modified_count)

# update_one(): mengubah status menjadi "Habis" khusus untuk 2 menu pilihan
koleksi_menu.update_one({"nama_menu": "Espresso"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Mie Goreng Pedas"}, {"$set": {"status": "Habis"}})

# Verifikasi hasil update
print("Jumlah status 'Tersedia':", koleksi_menu.count_documents({"status": "Tersedia"}))
print("Jumlah status 'Habis'   :", koleksi_menu.count_documents({"status": "Habis"}))
print("Menu yang habis:", [m["nama_menu"] for m in koleksi_menu.find({"status": "Habis"})])

Dokumen yang diberi status 'Tersedia': 14
Jumlah status 'Tersedia': 12
Jumlah status 'Habis'   : 2
Menu yang habis: ['Espresso', 'Mie Goreng Pedas']


## D. Integrasi ke Spark

Mengikuti pola Subbab 9.6 (PyMongo ke pandas ke Spark), dengan tambahan penanganan field yang kosong pada sebagian dokumen.

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, round as spark_round
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas9-MongoDBSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: ambil seluruh dokumen dari MongoDB, lalu jadikan pandas DataFrame
dokumen_menu = list(koleksi_menu.find({}))
df_pandas = pd.DataFrame(dokumen_menu)

# Langkah 2: _id bertipe ObjectId tidak dikenali Spark, ubah menjadi string
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Langkah 3: field yang tidak dimiliki sebuah dokumen menjadi NaN di pandas.
# NaN diganti None agar Spark membacanya sebagai null yang sebenarnya
df_pandas = df_pandas.astype(object).where(df_pandas.notna(), None)

# Langkah 4: konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

# Langkah 5: kolom skala yang tadinya desimal (5.0) dikembalikan menjadi bilangan bulat
for kolom in ["tingkat_kepahitan", "pedas_level", "tingkat_manis"]:
    df_spark_menu = df_spark_menu.withColumn(kolom, col(kolom).cast("int"))

print("Jumlah baris Spark DataFrame:", df_spark_menu.count())
df_spark_menu.select("nama_menu", "kategori", "tingkat_kepahitan", "pedas_level", "tingkat_manis").show(14)

Jumlah baris Spark DataFrame: 14
+--------------------+--------+-----------------+-----------+-------------+
|           nama_menu|kategori|tingkat_kepahitan|pedas_level|tingkat_manis|
+--------------------+--------+-----------------+-----------+-------------+
|            Espresso|    Kopi|                5|       NULL|         NULL|
|           Americano|    Kopi|                4|       NULL|         NULL|
|          Cappuccino|    Kopi|                3|       NULL|         NULL|
| Kopi Susu Gula Aren|    Kopi|                2|       NULL|         NULL|
|               Latte|    Kopi|                2|       NULL|         NULL|
|         Kopi Tubruk|    Kopi|                5|       NULL|         NULL|
|          Roti Bakar| Makanan|             NULL|          0|         NULL|
|      Kentang Goreng| Makanan|             NULL|          0|         NULL|
|    Mie Goreng Pedas| Makanan|             NULL|          3|         NULL|
|Nasi Ayam Sambal ...| Makanan|             NULL|      

In [9]:
# D.1 Skema hasil konversi
df_spark_menu.printSchema()

# Bukti bahwa field yang tidak ada terbaca sebagai null (bukan NaN)
print("Menu tanpa tingkat_kepahitan:", df_spark_menu.filter(col("tingkat_kepahitan").isNull()).count())
print("Menu tanpa pedas_level      :", df_spark_menu.filter(col("pedas_level").isNull()).count())

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: integer (nullable = true)
 |-- stok: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: integer (nullable = true)
 |-- tingkat_manis: integer (nullable = true)

Menu tanpa tingkat_kepahitan: 8
Menu tanpa pedas_level      : 9


In [11]:
# D.2 Rata-rata harga per kategori
df_spark_menu.groupBy("kategori").agg(
    spark_round(avg("harga"), 2).alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).orderBy("kategori").show()

+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
|    Kopi|       21166.67|          6|
| Makanan|        22400.0|          5|
|Non-Kopi|       21333.33|          3|
+--------+---------------+-----------+



## E. Kesimpulan

Tantangan utama saat mengonversi data MongoDB ke Spark DataFrame adalah ketidakcocokan antara dokumen yang skemanya fleksibel dengan DataFrame yang menuntut skema seragam. Pertama, field yang tidak dimiliki sebuah dokumen, seperti `tingkat_kepahitan` pada menu Makanan, menjadi NaN saat dimuat ke pandas. Pada pengaturan bawaan, NaN ini terbaca Spark sebagai nilai NaN, bukan `null`, sehingga fungsi seperti `isNull()` tidak mengenalinya. Kedua, kolom yang memuat NaN otomatis berubah menjadi tipe desimal, misalnya kepahitan 5 menjadi 5.0, padahal aslinya bilangan bulat. Ketiga, field khusus MongoDB seperti `_id` bertipe `ObjectId` tidak dikenali Spark. Saya mengatasinya dengan mengubah `_id` menjadi string, mengganti NaN dengan `None` lewat `astype(object)` dan `where`, lalu mengembalikan tipe kolom skala menjadi integer dengan `cast`. Hasilnya, kolom yang tidak dimiliki dokumen terbaca sebagai `null` yang benar, skema terbentuk dengan sepuluh kolom, dan `groupBy` rata rata harga berjalan tanpa error.

In [12]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
